## PROSES 0: Import library dan pengaturan
`SEED = 8` sama dengan yang dipakai pada hitungan manual, sehingga bootstrap dan fitur tiap pohon identik.

In [53]:
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display

SEED = 8          # seed sama dengan hitungan manual
N_POHON = 5       # jumlah pohon
N_LATIH = 10      # jumlah data latih untuk hitungan manual

PATH_LATIH = "data/data_latih_proses.csv"
PATH_UJI = "data/data_uji_proses.csv"

## PROSES 1: Load data dan ringkasan

In [54]:
df_latih = pd.read_csv(PATH_LATIH)
df_uji = pd.read_csv(PATH_UJI)

print("Data latih :", df_latih.shape)
print("Data uji   :", df_uji.shape)
print("Missing value (latih / uji):", df_latih.isna().sum().sum(), "/", df_uji.isna().sum().sum())
print("Baris duplikat pada data latih:", df_latih.duplicated().sum())
print("index_asli data latih unik?   :", df_latih["index_asli"].is_unique)
print("Ada index_asli yang sama antara latih dan uji?:", len(set(df_latih["index_asli"]) & set(df_uji["index_asli"])) > 0)

distribusi = pd.DataFrame({"Latih": df_latih["pass_status"].value_counts(),
                           "Uji": df_uji["pass_status"].value_counts()})
distribusi.loc["Total"] = distribusi.sum()
print("\nDistribusi kelas:")
display(distribusi)
print("Proporsi kelas data uji (%):")
print((df_uji["pass_status"].value_counts(normalize=True) * 100).round(2).to_string())

display(df_latih.head())

Data latih : (102912, 15)
Data uji   : (16605, 15)
Missing value (latih / uji): 0 / 0
Baris duplikat pada data latih: 36900
index_asli data latih unik?   : False
Ada index_asli yang sama antara latih dan uji?: False

Distribusi kelas:


,Latih,Uji
pass_status,,
Pass,51456,12816
Fail,51456,3789
Total,102912,16605


Proporsi kelas data uji (%):
pass_status
Pass    77.18
Fail    22.82


,index_asli,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
0,33919,59.68,2.34,85.61,76.83,3.76,2.41,6,26,6.54,7,7.17,1,4,Pass
1,44758,89.69,3.96,70.09,40.40,3.06,2.05,5,30,5.19,8,9.38,3,4,Pass
2,77911,55.98,2.16,75.46,76.48,1.86,1.56,5,11,8.93,10,8.82,3,2,Fail
3,78174,74.53,3.03,100.00,80.71,3.78,2.09,5,16,8.76,6,9.13,1,4,Pass
4,72558,56.88,2.29,69.01,76.03,4.36,3.55,5,5,5.39,10,9.46,3,4,Fail


## PROSES 2: Deskripsi variabel (X1 sampai X13 dan Y)
`index_asli` hanya penomoran sehingga **bukan fitur**. Fitur ada 13 dan targetnya `pass_status`.
(Deskripsi diturunkan dari nama kolom; cocokkan dengan dokumentasi dataset sumbermu.)

In [55]:
FITUR_ASLI = [c for c in df_latih.columns if c not in ("index_asli", "pass_status")]
LABEL_X = [f"X{i}" for i in range(1, len(FITUR_ASLI) + 1)]
PETA = dict(zip(FITUR_ASLI, LABEL_X))        # nama asli -> X
NAMA_ASLI = dict(zip(LABEL_X, FITUR_ASLI))   # X -> nama asli

deskripsi = {
    "previous_exam_score": ("Nilai ujian sebelumnya", "Numerik kontinu"),
    "previous_gpa": ("IPK sebelumnya", "Numerik kontinu"),
    "attendance_percentage": ("Persentase kehadiran", "Numerik kontinu"),
    "assignment_completion_rate": ("Persentase tugas yang diselesaikan", "Numerik kontinu"),
    "study_hours_per_day": ("Jam belajar per hari", "Numerik kontinu"),
    "self_study_hours": ("Jam belajar mandiri", "Numerik kontinu"),
    "practice_tests_completed": ("Jumlah tes latihan yang diselesaikan", "Numerik diskret"),
    "exam_preparation_days": ("Jumlah hari persiapan ujian", "Numerik diskret"),
    "sleep_hours": ("Jam tidur", "Numerik kontinu"),
    "stress_level": ("Tingkat stres", "Numerik ordinal"),
    "exam_anxiety_level": ("Tingkat kecemasan ujian", "Numerik kontinu"),
    "exam_difficulty": ("Tingkat kesulitan ujian (1=Easy, 2=Medium, 3=Hard)", "Numerik ordinal"),
    "family_income": ("Tingkat pendapatan keluarga (1=Low, 2=Lower-Middle, 3=Middle, 4=Upper-Middle, 5=High)", "Numerik ordinal"),
}

tabel_var = pd.DataFrame({
    "Label": LABEL_X,
    "Nama kolom": FITUR_ASLI,
    "Deskripsi": [deskripsi[c][0] for c in FITUR_ASLI],
    "Tipe": [deskripsi[c][1] for c in FITUR_ASLI],
    "Min (latih)": [df_latih[c].min() for c in FITUR_ASLI],
    "Max (latih)": [df_latih[c].max() for c in FITUR_ASLI],
})
tabel_var.loc[len(tabel_var)] = ["Y", "pass_status", "Status kelulusan (target)", "Kategorikal biner", "Fail", "Pass"]
display(tabel_var.set_index("Label"))

,Nama kolom,Deskripsi,Tipe,Min (latih),Max (latih)
Label,,,,,
X1,previous_exam_score,Nilai ujian sebelumnya,Numerik kontinu,30.0,100.0
X2,previous_gpa,IPK sebelumnya,Numerik kontinu,1.0,4.0
X3,attendance_percentage,Persentase kehadiran,Numerik kontinu,46.34,100.0
X4,assignment_completion_rate,Persentase tugas yang diselesaikan,Numerik kontinu,20.0,100.0
X5,study_hours_per_day,Jam belajar per hari,Numerik kontinu,0.5,7.195
X6,self_study_hours,Jam belajar mandiri,Numerik kontinu,0.25,5.095
X7,practice_tests_completed,Jumlah tes latihan yang diselesaikan,Numerik diskret,0.0,11.0
X8,exam_preparation_days,Jumlah hari persiapan ujian,Numerik diskret,1.0,30.0
X9,sleep_hours,Jam tidur,Numerik kontinu,3.0,11.0


## PROSES 3: Ambil 10 data latih dan 1 data uji
10 baris pertama data latih dan baris pertama data uji sama dengan data pada hitungan manual. Kolom diganti menjadi X1 sampai X13 dan Y, baris diberi nomor 1 sampai 10 (data uji nomor 11). Data uji **tidak ikut diundi**, dan labelnya baru dibuka saat evaluasi.

In [56]:
KOLOM_RENAME = {**PETA, "pass_status": "Y"}

def siapkan(df_sumber, baris, nomor):
    out = df_sumber.iloc[baris].drop(columns="index_asli").rename(columns=KOLOM_RENAME).copy()
    out.index = nomor
    out.index.name = "No"
    return out

latih10 = siapkan(df_latih, list(range(N_LATIH)), range(1, N_LATIH + 1))
uji1 = siapkan(df_uji, [0], [N_LATIH + 1])

uji_fitur = uji1.drop(columns="Y")       # yang dilihat model
label_asli = uji1["Y"].iloc[0]           # disimpan, dibuka saat evaluasi

print("DATA LATIH (10 baris)")
display(latih10)
print("DATA UJI (label disembunyikan)")
display(uji_fitur)

# pastikan sama dengan hitungan manual
assert latih10["Y"].tolist() == ["Pass","Pass","Fail","Pass","Fail","Fail","Fail","Fail","Pass","Fail"]
assert latih10["X5"].tolist() == [3.76, 3.06, 1.86, 3.78, 4.36, 0.93, 4.46, 2.65, 6.42, 2.7]
assert uji_fitur.iloc[0]["X5"] == 0.89 and label_asli == "Fail"
print("Data 10 latih + 1 uji sama dengan hitungan manual")
print(latih10["Y"].value_counts().to_string())

DATA LATIH (10 baris)


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,Y
No,,,,,,,,,,,,,,
1,59.68,2.34,85.61,76.83,3.76,2.41,6,26,6.54,7,7.17,1,4,Pass
2,89.69,3.96,70.09,40.40,3.06,2.05,5,30,5.19,8,9.38,3,4,Pass
3,55.98,2.16,75.46,76.48,1.86,1.56,5,11,8.93,10,8.82,3,2,Fail
4,74.53,3.03,100.00,80.71,3.78,2.09,5,16,8.76,6,9.13,1,4,Pass
5,56.88,2.29,69.01,76.03,4.36,3.55,5,5,5.39,10,9.46,3,4,Fail
6,75.80,3.11,64.24,42.00,0.93,0.56,8,10,7.37,10,10.00,3,1,Fail
7,50.96,2.14,88.46,80.58,4.46,3.51,4,6,7.12,8,10.00,2,2,Fail
8,56.75,2.33,83.80,58.49,2.65,1.92,3,1,6.80,8,6.06,3,1,Fail
9,47.22,1.85,69.20,89.50,6.42,3.40,4,18,6.32,10,10.00,1,2,Pass


DATA UJI (label disembunyikan)


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13
No,,,,,,,,,,,,,
11,83.38,3.45,77.92,76.64,0.89,0.55,4,23,5.1,10,10.0,2,1


Data 10 latih + 1 uji sama dengan hitungan manual
Y
Fail    6
Pass    4


## PROSES 4: Bootstrap sampling
Untuk tiap pohon, ambil 10 index baris secara acak **dengan pengembalian** dari nomor 1 sampai 10. Index yang sama boleh muncul lebih dari sekali, dan index yang tidak terambil disebut data OOB (*out-of-bag*).

In [57]:
rng = np.random.default_rng(SEED)

bootstrap = {t: rng.choice(latih10.index.to_numpy(), size=N_LATIH, replace=True)
             for t in range(1, N_POHON + 1)}
bootstrap_urut = {t: sorted(int(i) for i in idx) for t, idx in bootstrap.items()}

ringkas = []
for t, idx in bootstrap_urut.items():
    y = latih10.loc[idx, "Y"]
    ringkas.append({
        "Pohon": t,
        "Index terambil": ", ".join(map(str, idx)),
        "Pass": int((y == "Pass").sum()),
        "Fail": int((y == "Fail").sum()),
        "OOB (tidak terambil)": ", ".join(str(i) for i in latih10.index if i not in idx),
    })
print("Hasil bootstrap")
display(pd.DataFrame(ringkas).set_index("Pohon"))

tally = pd.DataFrame({t: pd.Series(idx).value_counts().reindex(latih10.index, fill_value=0)
                      for t, idx in bootstrap_urut.items()}).T
tally["Total"] = tally.sum(axis=1)
tally.index.name = "Pohon"
print("Tabel hitung (berapa kali tiap index terambil)")
display(tally)

# sama dengan hitungan manual
harapan = {1: [2,3,4,4,7,7,8,8,9,10], 2: [1,1,2,4,4,4,5,5,6,6], 3: [2,2,2,3,3,4,9,9,9,10],
           4: [1,3,4,5,5,5,6,7,7,9], 5: [1,1,2,3,4,6,9,9,9,10]}
assert bootstrap_urut == harapan
print("Bootstrap sama dengan hitungan manual")

Hasil bootstrap


,Index terambil,Pass,Fail,OOB (tidak terambil)
Pohon,,,,
1,"2, 3, 4, 4, 7, 7, 8, 8, 9, 10",4,6,"1, 5, 6"
2,"1, 1, 2, 4, 4, 4, 5, 5, 6, 6",6,4,"3, 7, 8, 9, 10"
3,"2, 2, 2, 3, 3, 4, 9, 9, 9, 10",7,3,"1, 5, 6, 7, 8"
4,"1, 3, 4, 5, 5, 5, 6, 7, 7, 9",3,7,"2, 8, 10"
5,"1, 1, 2, 3, 4, 6, 9, 9, 9, 10",7,3,"5, 7, 8"


Tabel hitung (berapa kali tiap index terambil)


No,1,2,3,4,5,6,7,8,9,10,Total
Pohon,,,,,,,,,,,
1,0,1,1,2,0,0,2,2,1,1,10
2,2,1,0,3,2,2,0,0,0,0,10
3,0,3,2,1,0,0,0,0,3,1,10
4,1,0,1,1,3,1,2,0,1,0,10
5,2,1,1,1,0,1,0,0,3,1,10


Bootstrap sama dengan hitungan manual


## PROSES 5: Pilih 1 fitur acak untuk tiap pohon
Dari 13 fitur diundi 5 fitur **tanpa pengembalian** (satu fitur per pohon). Pemilihan ini tidak memakai Gini atau entropy, tetapi murni acak. Gini baru dipakai setelah fitur ditentukan, untuk mencari threshold terbaik.

In [58]:
pilih = rng.choice(LABEL_X, size=N_POHON, replace=False)
fitur_pohon = {t: str(f) for t, f in zip(range(1, N_POHON + 1), pilih)}

display(pd.DataFrame({
    "Pohon": list(fitur_pohon.keys()),
    "Fitur": list(fitur_pohon.values()),
    "Nama kolom": [NAMA_ASLI[f] for f in fitur_pohon.values()],
}).set_index("Pohon"))

tidak_terundi = [f for f in LABEL_X if f not in fitur_pohon.values()]
print("Fitur yang tidak terundi:", ", ".join(tidak_terundi))

assert fitur_pohon == {1: "X5", 2: "X4", 3: "X13", 4: "X12", 5: "X2"}
print("Fitur tiap pohon sama dengan hitungan manual")

,Fitur,Nama kolom
Pohon,,
1,X5,study_hours_per_day
2,X4,assignment_completion_rate
3,X13,family_income
4,X12,exam_difficulty
5,X2,previous_gpa


Fitur yang tidak terundi: X1, X3, X6, X7, X8, X9, X10, X11
Fitur tiap pohon sama dengan hitungan manual


## PROSES 6: Fungsi Gini dan threshold
- **Gini(S)** = 1 − (p_Pass² + p_Fail²)
- **Threshold** = titik tengah dua nilai unik yang berurutan: (nilai atas + nilai bawah) / 2
- **Gini total** = (n kiri / n) × Gini kiri + (n kanan / n) × Gini kanan, dan threshold dengan **Gini total terkecil** dipilih
- Prediksi tiap sisi = **kelas mayoritas** pada kelompok itu

In [59]:
def gini(labels):
    n = len(labels)
    if n == 0:
        return 0.0
    return 1 - sum((c / n) ** 2 for c in Counter(labels).values())

def fmt(x, d=4):
    return f"{x:.{d}f}"

def teks_gini(labels, nama="S"):
    """Teks rumus Gini, contoh: Gini(S) = 1 - ((4/10)^2 + (6/10)^2) = 0.48"""
    n = len(labels)
    p, f = labels.count("Pass"), labels.count("Fail")
    g = gini(labels)
    return f"Gini({nama}) = 1 − (({p}/{n})² + ({f}/{n})²) = {fmt(g)}", g

def kelas_mayoritas(labels, cadangan=None):
    c = Counter(labels).most_common()
    if len(c) > 1 and c[0][1] == c[1][1]:                  # seri: pakai kelas mayoritas seluruh sampel
        return kelas_mayoritas(cadangan) if cadangan else "Fail"
    return c[0][0]

def hitung_semua_split(sampel, fitur):
    """Hitung Gini semua calon threshold untuk 1 fitur."""
    n = len(sampel)
    g0 = gini(sampel["Y"].tolist())
    unik = sorted(sampel[fitur].unique())
    baris = []
    for a, b in zip(unik[:-1], unik[1:]):
        th = round((a + b) / 2, 6)
        kiri = sampel.loc[sampel[fitur] <= th, "Y"].tolist()
        kanan = sampel.loc[sampel[fitur] > th, "Y"].tolist()
        g1, g2 = gini(kiri), gini(kanan)
        total = len(kiri) / n * g1 + len(kanan) / n * g2
        baris.append({"Threshold": th,
                      "K1 n": len(kiri), "K1 Pass": kiri.count("Pass"), "K1 Fail": kiri.count("Fail"), "Gini K1": g1,
                      "K2 n": len(kanan), "K2 Pass": kanan.count("Pass"), "K2 Fail": kanan.count("Fail"), "Gini K2": g2,
                      "Gini total": total, "Reduction": g0 - total})
    return pd.DataFrame(baris)

def bangun_stump(sampel, fitur):
    """Pohon 1 fitur: cari threshold dengan Gini total terkecil, lalu tentukan prediksi tiap sisi."""
    hasil = hitung_semua_split(sampel, fitur)
    terbaik = hasil.loc[hasil["Gini total"].round(10).idxmin()]
    th = float(terbaik["Threshold"])
    semua = sampel["Y"].tolist()
    return {"fitur": fitur, "threshold": th,
            "pred_kiri": kelas_mayoritas(sampel.loc[sampel[fitur] <= th, "Y"].tolist(), semua),
            "pred_kanan": kelas_mayoritas(sampel.loc[sampel[fitur] > th, "Y"].tolist(), semua),
            "gini_total": float(terbaik["Gini total"])}

def prediksi_stump(aturan, nilai):
    return aturan["pred_kiri"] if nilai <= aturan["threshold"] else aturan["pred_kanan"]

def proses_pohon(t):
    """Tampilkan seluruh proses satu pohon, format sama dengan laporan."""
    fitur = fitur_pohon[t]
    idx = bootstrap_urut[t]
    sampel = latih10.loc[idx, [fitur, "Y"]].copy()
    n = len(sampel)

    print("=" * 72)
    print(f"POHON {t} | fitur {fitur} ({NAMA_ASLI[fitur]})")
    print("=" * 72)

    # 1. sampel bootstrap
    print("\n1. Sampel bootstrap (index):", idx)
    tabel = sampel.copy()
    tabel.insert(0, "Index", tabel.index)
    tabel.index = range(1, n + 1)
    tabel.index.name = "No"
    display(tabel)

    # 2. data diurutkan
    print(f"2. Data diurutkan berdasarkan {fitur}")
    urut = sampel.sort_values(fitur, kind="stable")
    display(pd.DataFrame([urut[fitur].tolist(), urut["Y"].tolist()], index=[fitur, "Y"], columns=range(1, n + 1)))

    # 3. gini awal
    teks, g0 = teks_gini(sampel["Y"].tolist(), "S")
    print("3. Gini awal (root node)")
    print("   Jumlah Pass =", sampel["Y"].tolist().count("Pass"), ", Fail =", sampel["Y"].tolist().count("Fail"))
    print("   " + teks)

    # 4. threshold
    unik = sorted(sampel[fitur].unique())
    print(f"\n4. Threshold: {len(unik)} nilai unik menghasilkan {len(unik) - 1} threshold")
    display(pd.DataFrame([{"Pasangan nilai": f"{a:g} dan {b:g}",
                           "Perhitungan": f"({a:g} + {b:g}) / 2",
                           "Threshold": round((a + b) / 2, 6)} for a, b in zip(unik[:-1], unik[1:])]))

    # 5. hasil semua threshold
    hasil = hitung_semua_split(sampel, fitur)
    aturan_t = bangun_stump(sampel, fitur)
    th = aturan_t["threshold"]

    # 6. contoh perhitungan pada threshold terpilih
    kiri = sampel.loc[sampel[fitur] <= th, "Y"].tolist()
    kanan = sampel.loc[sampel[fitur] > th, "Y"].tolist()
    t1, g1 = teks_gini(kiri, "K1")
    t2, g2 = teks_gini(kanan, "K2")
    total = len(kiri) / n * g1 + len(kanan) / n * g2
    print(f"5. Contoh perhitungan Gini: uji split {fitur} ≤ {th:g} (threshold terpilih)")
    print(f"   Kelompok 1 ({fitur} ≤ {th:g}) → {kiri}")
    print("   " + t1)
    print(f"   Kelompok 2 ({fitur} > {th:g}) → {kanan}")
    print("   " + t2)
    print(f"   Gini total = ({len(kiri)}/{n})({fmt(g1)}) + ({len(kanan)}/{n})({fmt(g2)}) = {fmt(total)}")
    print(f"   Reduction  = {fmt(g0)} − {fmt(total)} = {fmt(g0 - total)}")

    # 7. tabel hasil
    print("\n6. Tabel hasil semua threshold (Gini total terkecil dipilih)")
    tampil = hasil.copy()
    tampil["Terpilih"] = np.where(tampil["Threshold"] == th, "◄ terkecil", "")
    display(tampil.round(4))

    # 8. rule
    print("7. Rule (aturan) pohon")
    print(f"   {fitur} ≤ {th:g}  →  {aturan_t['pred_kiri']}   (isi: {kiri.count('Pass')} Pass, {kiri.count('Fail')} Fail)")
    print(f"   {fitur} > {th:g}  →  {aturan_t['pred_kanan']}   (isi: {kanan.count('Pass')} Pass, {kanan.count('Fail')} Fail)")
    return aturan_t

aturan = {}

## PROSES 7.1: Pohon 1 (fitur X5 / study_hours_per_day)

In [60]:
aturan[1] = proses_pohon(1)

POHON 1 | fitur X5 (study_hours_per_day)

1. Sampel bootstrap (index): [2, 3, 4, 4, 7, 7, 8, 8, 9, 10]


,Index,X5,Y
No,,,
1,2,3.06,Pass
2,3,1.86,Fail
3,4,3.78,Pass
4,4,3.78,Pass
5,7,4.46,Fail
6,7,4.46,Fail
7,8,2.65,Fail
8,8,2.65,Fail
9,9,6.42,Pass


2. Data diurutkan berdasarkan X5


,1,2,3,4,5,6,7,8,9,10
X5,1.86,2.65,2.65,2.7,3.06,3.78,3.78,4.46,4.46,6.42
Y,Fail,Fail,Fail,Fail,Pass,Pass,Pass,Fail,Fail,Pass


3. Gini awal (root node)
   Jumlah Pass = 4 , Fail = 6
   Gini(S) = 1 − ((4/10)² + (6/10)²) = 0.4800

4. Threshold: 7 nilai unik menghasilkan 6 threshold


,Pasangan nilai,Perhitungan,Threshold
0,1.86 dan 2.65,(1.86 + 2.65) / 2,2.255
1,2.65 dan 2.7,(2.65 + 2.7) / 2,2.675
2,2.7 dan 3.06,(2.7 + 3.06) / 2,2.880
3,3.06 dan 3.78,(3.06 + 3.78) / 2,3.420
4,3.78 dan 4.46,(3.78 + 4.46) / 2,4.120
5,4.46 dan 6.42,(4.46 + 6.42) / 2,5.440


5. Contoh perhitungan Gini: uji split X5 ≤ 2.88 (threshold terpilih)
   Kelompok 1 (X5 ≤ 2.88) → ['Fail', 'Fail', 'Fail', 'Fail']
   Gini(K1) = 1 − ((0/4)² + (4/4)²) = 0.0000
   Kelompok 2 (X5 > 2.88) → ['Pass', 'Pass', 'Pass', 'Fail', 'Fail', 'Pass']
   Gini(K2) = 1 − ((4/6)² + (2/6)²) = 0.4444
   Gini total = (4/10)(0.0000) + (6/10)(0.4444) = 0.2667
   Reduction  = 0.4800 − 0.2667 = 0.2133

6. Tabel hasil semua threshold (Gini total terkecil dipilih)


,Threshold,K1 n,K1 Pass,K1 Fail,Gini K1,K2 n,K2 Pass,K2 Fail,Gini K2,Gini total,Reduction,Terpilih
0,2.255,1,0,1,0.0000,9,4,5,0.4938,0.4444,0.0356,
1,2.675,3,0,3,0.0000,7,4,3,0.4898,0.3429,0.1371,
2,2.880,4,0,4,0.0000,6,4,2,0.4444,0.2667,0.2133,◄ terkecil
3,3.420,5,1,4,0.3200,5,3,2,0.4800,0.4000,0.0800,
4,4.120,7,3,4,0.4898,3,1,2,0.4444,0.4762,0.0038,
5,5.440,9,3,6,0.4444,1,1,0,0.0000,0.4000,0.0800,


7. Rule (aturan) pohon
   X5 ≤ 2.88  →  Fail   (isi: 0 Pass, 4 Fail)
   X5 > 2.88  →  Pass   (isi: 4 Pass, 2 Fail)


## PROSES 7.2: Pohon 2 (fitur X4 / assignment_completion_rate)

In [61]:
aturan[2] = proses_pohon(2)

POHON 2 | fitur X4 (assignment_completion_rate)

1. Sampel bootstrap (index): [1, 1, 2, 4, 4, 4, 5, 5, 6, 6]


,Index,X4,Y
No,,,
1,1,76.83,Pass
2,1,76.83,Pass
3,2,40.40,Pass
4,4,80.71,Pass
5,4,80.71,Pass
6,4,80.71,Pass
7,5,76.03,Fail
8,5,76.03,Fail
9,6,42.00,Fail


2. Data diurutkan berdasarkan X4


,1,2,3,4,5,6,7,8,9,10
X4,40.4,42.0,42.0,76.03,76.03,76.83,76.83,80.71,80.71,80.71
Y,Pass,Fail,Fail,Fail,Fail,Pass,Pass,Pass,Pass,Pass


3. Gini awal (root node)
   Jumlah Pass = 6 , Fail = 4
   Gini(S) = 1 − ((6/10)² + (4/10)²) = 0.4800

4. Threshold: 5 nilai unik menghasilkan 4 threshold


,Pasangan nilai,Perhitungan,Threshold
0,40.4 dan 42,(40.4 + 42) / 2,41.200
1,42 dan 76.03,(42 + 76.03) / 2,59.015
2,76.03 dan 76.83,(76.03 + 76.83) / 2,76.430
3,76.83 dan 80.71,(76.83 + 80.71) / 2,78.770


5. Contoh perhitungan Gini: uji split X4 ≤ 76.43 (threshold terpilih)
   Kelompok 1 (X4 ≤ 76.43) → ['Pass', 'Fail', 'Fail', 'Fail', 'Fail']
   Gini(K1) = 1 − ((1/5)² + (4/5)²) = 0.3200
   Kelompok 2 (X4 > 76.43) → ['Pass', 'Pass', 'Pass', 'Pass', 'Pass']
   Gini(K2) = 1 − ((5/5)² + (0/5)²) = 0.0000
   Gini total = (5/10)(0.3200) + (5/10)(0.0000) = 0.1600
   Reduction  = 0.4800 − 0.1600 = 0.3200

6. Tabel hasil semua threshold (Gini total terkecil dipilih)


,Threshold,K1 n,K1 Pass,K1 Fail,Gini K1,K2 n,K2 Pass,K2 Fail,Gini K2,Gini total,Reduction,Terpilih
0,41.200,1,1,0,0.0000,9,5,4,0.4938,0.4444,0.0356,
1,59.015,3,1,2,0.4444,7,5,2,0.4082,0.4190,0.0610,
2,76.430,5,1,4,0.3200,5,5,0,0.0000,0.1600,0.3200,◄ terkecil
3,78.770,7,3,4,0.4898,3,3,0,0.0000,0.3429,0.1371,


7. Rule (aturan) pohon
   X4 ≤ 76.43  →  Fail   (isi: 1 Pass, 4 Fail)
   X4 > 76.43  →  Pass   (isi: 5 Pass, 0 Fail)


## PROSES 7.3: Pohon 3 (fitur X13 / family_income)

In [62]:
aturan[3] = proses_pohon(3)

POHON 3 | fitur X13 (family_income)

1. Sampel bootstrap (index): [2, 2, 2, 3, 3, 4, 9, 9, 9, 10]


,Index,X13,Y
No,,,
1,2,4,Pass
2,2,4,Pass
3,2,4,Pass
4,3,2,Fail
5,3,2,Fail
6,4,4,Pass
7,9,2,Pass
8,9,2,Pass
9,9,2,Pass


2. Data diurutkan berdasarkan X13


,1,2,3,4,5,6,7,8,9,10
X13,2,2,2,2,2,4,4,4,4,5
Y,Fail,Fail,Pass,Pass,Pass,Pass,Pass,Pass,Pass,Fail


3. Gini awal (root node)
   Jumlah Pass = 7 , Fail = 3
   Gini(S) = 1 − ((7/10)² + (3/10)²) = 0.4200

4. Threshold: 3 nilai unik menghasilkan 2 threshold


,Pasangan nilai,Perhitungan,Threshold
0,2 dan 4,(2 + 4) / 2,3.0
1,4 dan 5,(4 + 5) / 2,4.5


5. Contoh perhitungan Gini: uji split X13 ≤ 4.5 (threshold terpilih)
   Kelompok 1 (X13 ≤ 4.5) → ['Pass', 'Pass', 'Pass', 'Fail', 'Fail', 'Pass', 'Pass', 'Pass', 'Pass']
   Gini(K1) = 1 − ((7/9)² + (2/9)²) = 0.3457
   Kelompok 2 (X13 > 4.5) → ['Fail']
   Gini(K2) = 1 − ((0/1)² + (1/1)²) = 0.0000
   Gini total = (9/10)(0.3457) + (1/10)(0.0000) = 0.3111
   Reduction  = 0.4200 − 0.3111 = 0.1089

6. Tabel hasil semua threshold (Gini total terkecil dipilih)


,Threshold,K1 n,K1 Pass,K1 Fail,Gini K1,K2 n,K2 Pass,K2 Fail,Gini K2,Gini total,Reduction,Terpilih
0,3.0,5,3,2,0.4800,5,4,1,0.32,0.4000,0.0200,
1,4.5,9,7,2,0.3457,1,0,1,0.00,0.3111,0.1089,◄ terkecil


7. Rule (aturan) pohon
   X13 ≤ 4.5  →  Pass   (isi: 7 Pass, 2 Fail)
   X13 > 4.5  →  Fail   (isi: 0 Pass, 1 Fail)


## PROSES 7.4: Pohon 4 (fitur X12 / exam_difficulty)

In [63]:
aturan[4] = proses_pohon(4)

POHON 4 | fitur X12 (exam_difficulty)

1. Sampel bootstrap (index): [1, 3, 4, 5, 5, 5, 6, 7, 7, 9]


,Index,X12,Y
No,,,
1,1,1,Pass
2,3,3,Fail
3,4,1,Pass
4,5,3,Fail
5,5,3,Fail
6,5,3,Fail
7,6,3,Fail
8,7,2,Fail
9,7,2,Fail


2. Data diurutkan berdasarkan X12


,1,2,3,4,5,6,7,8,9,10
X12,1,1,1,2,2,3,3,3,3,3
Y,Pass,Pass,Pass,Fail,Fail,Fail,Fail,Fail,Fail,Fail


3. Gini awal (root node)
   Jumlah Pass = 3 , Fail = 7
   Gini(S) = 1 − ((3/10)² + (7/10)²) = 0.4200

4. Threshold: 3 nilai unik menghasilkan 2 threshold


,Pasangan nilai,Perhitungan,Threshold
0,1 dan 2,(1 + 2) / 2,1.5
1,2 dan 3,(2 + 3) / 2,2.5


5. Contoh perhitungan Gini: uji split X12 ≤ 1.5 (threshold terpilih)
   Kelompok 1 (X12 ≤ 1.5) → ['Pass', 'Pass', 'Pass']
   Gini(K1) = 1 − ((3/3)² + (0/3)²) = 0.0000
   Kelompok 2 (X12 > 1.5) → ['Fail', 'Fail', 'Fail', 'Fail', 'Fail', 'Fail', 'Fail']
   Gini(K2) = 1 − ((0/7)² + (7/7)²) = 0.0000
   Gini total = (3/10)(0.0000) + (7/10)(0.0000) = 0.0000
   Reduction  = 0.4200 − 0.0000 = 0.4200

6. Tabel hasil semua threshold (Gini total terkecil dipilih)


,Threshold,K1 n,K1 Pass,K1 Fail,Gini K1,K2 n,K2 Pass,K2 Fail,Gini K2,Gini total,Reduction,Terpilih
0,1.5,3,3,0,0.00,7,0,7,0.0,0.00,0.42,◄ terkecil
1,2.5,5,3,2,0.48,5,0,5,0.0,0.24,0.18,


7. Rule (aturan) pohon
   X12 ≤ 1.5  →  Pass   (isi: 3 Pass, 0 Fail)
   X12 > 1.5  →  Fail   (isi: 0 Pass, 7 Fail)


## PROSES 7.5: Pohon 5 (fitur X2 / previous_gpa)

In [64]:
aturan[5] = proses_pohon(5)

POHON 5 | fitur X2 (previous_gpa)

1. Sampel bootstrap (index): [1, 1, 2, 3, 4, 6, 9, 9, 9, 10]


,Index,X2,Y
No,,,
1,1,2.34,Pass
2,1,2.34,Pass
3,2,3.96,Pass
4,3,2.16,Fail
5,4,3.03,Pass
6,6,3.11,Fail
7,9,1.85,Pass
8,9,1.85,Pass
9,9,1.85,Pass


2. Data diurutkan berdasarkan X2


,1,2,3,4,5,6,7,8,9,10
X2,1.85,1.85,1.85,2.16,2.34,2.34,3.03,3.11,3.25,3.96
Y,Pass,Pass,Pass,Fail,Pass,Pass,Pass,Fail,Fail,Pass


3. Gini awal (root node)
   Jumlah Pass = 7 , Fail = 3
   Gini(S) = 1 − ((7/10)² + (3/10)²) = 0.4200

4. Threshold: 7 nilai unik menghasilkan 6 threshold


,Pasangan nilai,Perhitungan,Threshold
0,1.85 dan 2.16,(1.85 + 2.16) / 2,2.005
1,2.16 dan 2.34,(2.16 + 2.34) / 2,2.250
2,2.34 dan 3.03,(2.34 + 3.03) / 2,2.685
3,3.03 dan 3.11,(3.03 + 3.11) / 2,3.070
4,3.11 dan 3.25,(3.11 + 3.25) / 2,3.180
5,3.25 dan 3.96,(3.25 + 3.96) / 2,3.605


5. Contoh perhitungan Gini: uji split X2 ≤ 3.07 (threshold terpilih)
   Kelompok 1 (X2 ≤ 3.07) → ['Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Pass', 'Pass']
   Gini(K1) = 1 − ((6/7)² + (1/7)²) = 0.2449
   Kelompok 2 (X2 > 3.07) → ['Pass', 'Fail', 'Fail']
   Gini(K2) = 1 − ((1/3)² + (2/3)²) = 0.4444
   Gini total = (7/10)(0.2449) + (3/10)(0.4444) = 0.3048
   Reduction  = 0.4200 − 0.3048 = 0.1152

6. Tabel hasil semua threshold (Gini total terkecil dipilih)


,Threshold,K1 n,K1 Pass,K1 Fail,Gini K1,K2 n,K2 Pass,K2 Fail,Gini K2,Gini total,Reduction,Terpilih
0,2.005,3,3,0,0.0000,7,4,3,0.4898,0.3429,0.0771,
1,2.250,4,3,1,0.3750,6,4,2,0.4444,0.4167,0.0033,
2,2.685,6,5,1,0.2778,4,2,2,0.5000,0.3667,0.0533,
3,3.070,7,6,1,0.2449,3,1,2,0.4444,0.3048,0.1152,◄ terkecil
4,3.180,8,6,2,0.3750,2,1,1,0.5000,0.4000,0.0200,
5,3.605,9,6,3,0.4444,1,1,0,0.0000,0.4000,0.0200,


7. Rule (aturan) pohon
   X2 ≤ 3.07  →  Pass   (isi: 6 Pass, 1 Fail)
   X2 > 3.07  →  Fail   (isi: 1 Pass, 2 Fail)


### Rekap rule kelima pohon

In [65]:
rekap = pd.DataFrame([{"Pohon": t, "Fitur": a["fitur"], "Nama kolom": NAMA_ASLI[a["fitur"]],
                       "Threshold": a["threshold"], "Gini total": round(a["gini_total"], 4),
                       "Rule kiri (≤)": a["pred_kiri"], "Rule kanan (>)": a["pred_kanan"]}
                      for t, a in aturan.items()]).set_index("Pohon")
display(rekap)

# sama dengan hitungan manual
manual = {1: (2.88, "Fail", "Pass"), 2: (76.43, "Fail", "Pass"), 3: (4.5, "Pass", "Fail"),
          4: (1.5, "Pass", "Fail"), 5: (3.07, "Pass", "Fail")}
for t, (th, kiri, kanan) in manual.items():
    a = aturan[t]
    assert np.isclose(a["threshold"], th) and a["pred_kiri"] == kiri and a["pred_kanan"] == kanan, t
print("Threshold dan rule kelima pohon sama dengan hitungan manual")

,Fitur,Nama kolom,Threshold,Gini total,Rule kiri (≤),Rule kanan (>)
Pohon,,,,,,
1,X5,study_hours_per_day,2.88,0.2667,Fail,Pass
2,X4,assignment_completion_rate,76.43,0.1600,Fail,Pass
3,X13,family_income,4.50,0.3111,Pass,Fail
4,X12,exam_difficulty,1.50,0.0000,Pass,Fail
5,X2,previous_gpa,3.07,0.3048,Pass,Fail


Threshold dan rule kelima pohon sama dengan hitungan manual


## PROSES 8: Pengujian data uji dan voting
Tiap pohon hanya melihat **satu kolom**, yaitu fitur miliknya. Label asli data uji baru dibuka setelah voting.

In [66]:
baris, suara = [], []
for t in range(1, N_POHON + 1):
    a = aturan[t]
    f = a["fitur"]
    nilai = float(uji_fitur.iloc[0][f])
    kiri = nilai <= a["threshold"]
    pred = prediksi_stump(a, nilai)
    suara.append(pred)
    baris.append({"Pohon": t, "Fitur": f"{f} ({NAMA_ASLI[f]})", "Nilai data uji": nilai,
                  "Threshold": a["threshold"],
                  "Perbandingan": f"{nilai:g} {'≤' if kiri else '>'} {a['threshold']:g}",
                  "Cabang": "Kiri" if kiri else "Kanan", "Prediksi": pred})
display(pd.DataFrame(baris).set_index("Pohon"))

hitung = Counter(suara)
hasil_akhir = hitung.most_common(1)[0][0]
print("Voting :", dict(hitung))
print("Prediksi akhir Random Forest :", hasil_akhir)
print("Label asli data uji          :", label_asli)
print("Hasil                        :", "BENAR" if hasil_akhir == label_asli else "SALAH")
print("Akurasi (1 data uji)         :", f"{(hasil_akhir == label_asli) * 100:.0f}%")

pohon_salah = [t for t, p in enumerate(suara, 1) if p != label_asli]
print("Pohon yang prediksinya keliru:", pohon_salah if pohon_salah else "tidak ada")

,Fitur,Nilai data uji,Threshold,Perbandingan,Cabang,Prediksi
Pohon,,,,,,
1,X5 (study_hours_per_day),0.89,2.88,0.89 ≤ 2.88,Kiri,Fail
2,X4 (assignment_completion_rate),76.64,76.43,76.64 > 76.43,Kanan,Pass
3,X13 (family_income),1.00,4.50,1 ≤ 4.5,Kiri,Pass
4,X12 (exam_difficulty),2.00,1.50,2 > 1.5,Kanan,Fail
5,X2 (previous_gpa),3.45,3.07,3.45 > 3.07,Kanan,Fail


Voting : {'Fail': 3, 'Pass': 2}
Prediksi akhir Random Forest : Fail
Label asli data uji          : Fail
Hasil                        : BENAR
Akurasi (1 data uji)         : 100%
Pohon yang prediksinya keliru: [2, 3]


## PROSES 11: Program 2, Random Forest dengan seluruh data
Memakai seluruh data latih dan seluruh data uji, 13 fitur. Dua konfigurasi:

- **A (mirip hitungan manual):** 5 pohon, `max_features=1`, `max_depth=1`
- **B (standar):** 100 pohon, `max_features="sqrt"`, kedalaman bebas

Catatan: data uji **tidak seimbang** (kelas Pass jauh lebih banyak), sedangkan data latih seimbang. Selain akurasi, perhatikan juga *recall* dan *F1* kelas Fail.

In [67]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, recall_score

X_latih, y_latih = df_latih[FITUR_ASLI], np.array(df_latih["pass_status"].tolist())
X_uji, y_uji = df_uji[FITUR_ASLI], np.array(df_uji["pass_status"].tolist())

def latih_dan_evaluasi(nama, model):
    t0 = time.time()
    model.fit(X_latih, y_latih)
    waktu = time.time() - t0
    pred = model.predict(X_uji)
    print("=" * 72)
    print(nama)
    print("=" * 72)
    print(f"Waktu latih : {waktu:.1f} detik")
    print(f"Akurasi     : {accuracy_score(y_uji, pred):.4f}")
    print("\nClassification report:")
    print(classification_report(y_uji, pred, digits=4))
    print("Confusion matrix:")
    display(pd.DataFrame(confusion_matrix(y_uji, pred, labels=["Fail", "Pass"]),
                         index=["Aktual Fail", "Aktual Pass"], columns=["Prediksi Fail", "Prediksi Pass"]))
    return {"Model": nama,
            "Akurasi": accuracy_score(y_uji, pred),
            "Recall Fail": recall_score(y_uji, pred, pos_label="Fail"),
            "F1 Fail": f1_score(y_uji, pred, pos_label="Fail"),
            "F1 Pass": f1_score(y_uji, pred, pos_label="Pass")}

model_A = RandomForestClassifier(n_estimators=5, max_features=1, max_depth=1, random_state=SEED, n_jobs=-1)
hasil_A = latih_dan_evaluasi("Konfigurasi A: 5 pohon, 1 fitur per node, kedalaman 1 (mirip manual)", model_A)

Konfigurasi A: 5 pohon, 1 fitur per node, kedalaman 1 (mirip manual)
Waktu latih : 0.2 detik
Akurasi     : 0.6483

Classification report:
              precision    recall  f1-score   support

        Fail     0.3669    0.7464    0.4920      3789
        Pass     0.8920    0.6193    0.7310     12816

    accuracy                         0.6483     16605
   macro avg     0.6295    0.6828    0.6115     16605
weighted avg     0.7722    0.6483    0.6765     16605

Confusion matrix:


,Prediksi Fail,Prediksi Pass
Aktual Fail,2828,961
Aktual Pass,4879,7937


Interpretasi hasil prediksi
2.828
Prediksi benar (Fail)

Dari seluruh data yang sebenarnya Fail, sebanyak 2.828 berhasil dikenali sebagai Fail.


961
Prediksi salah (Fail menjadi Pass)

Sebanyak 961 data yang sebenarnya Fail justru diprediksi sebagai Pass.


4.879
Prediksi salah (Pass menjadi Fail)

Sebanyak 4.879 data yang sebenarnya Pass justru diprediksi sebagai Fail.


7.937
Prediksi benar (Pass)

Sebanyak 7.937 data yang sebenarnya Pass berhasil dikenali sebagai Pass.

Kesimpulan jumlah prediksi
10.765
Benar
5.840
Salah
16.605
Total data uji

Metrik	Arti
Precision	Ketepatan prediksi model pada suatu kelas.
Recall	Kemampuan model menemukan seluruh data aktual pada suatu kelas.
F1-score	Gabungan keseimbangan Precision dan Recall.
Support	Jumlah data aktual pada masing-masing kelas.

Contohnya dari hasilmu:
- Fail – Recall 0.7464: Model berhasil menemukan 74,64% dari seluruh data yang sebenarnya Fail.
- Pass – Recall 0.6193: Model berhasil menemukan 61,93% dari seluruh data yang sebenarnya Pass.
- Accuracy 0.6483: Model menghasilkan prediksi benar sebesar 64,83% dari seluruh data pengujian.